# Train YOLO11 on a Custom Dataset in Kaggle

This notebook is adapted for Kaggle. It expects a dataset added through **Add Input** in YOLO format:

```text
dataset.zip or dataset folder
  data.yaml            # optional, this notebook rebuilds a clean one
  train/images/*
  train/labels/*
  valid/images/*       # or val/images/*
  valid/labels/*       # or val/labels/*
  test/images/*        # optional
  test/labels/*        # optional
```

Kaggle notes:
- Enable GPU in **Settings > Accelerator**.
- Enable Internet if `ultralytics` or `yolo11*.pt` must be downloaded.
- Outputs are written to `/kaggle/working`, where Kaggle lets you download them.


In [ ]:
!nvidia-smi


In [ ]:
%pip install -q -U "ultralytics>=8.3.0" pyyaml

import ultralytics
ultralytics.checks()


## Kaggle Dataset Configuration

Set `DATASET_INPUT_PATH` to the folder created by Kaggle under `/kaggle/input`. If you leave it as `None`, the notebook will scan `/kaggle/input` and use the first YOLO dataset or zip it can resolve.


In [ ]:
from pathlib import Path

# Example: DATASET_INPUT_PATH = "/kaggle/input/rada-tpii-yolo-dataset"
DATASET_INPUT_PATH = None

WORKDIR = Path("/kaggle/working")
KAGGLE_INPUT_DIR = Path("/kaggle/input")
EXTRACT_DIR = WORKDIR / "custom_dataset"
RESULTS_DIR = WORKDIR / "yolo11_training_results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Kaggle input dir exists:", KAGGLE_INPUT_DIR.exists())
print("Configured dataset input path:", DATASET_INPUT_PATH)
print("Results will be saved to:", RESULTS_DIR)

if KAGGLE_INPUT_DIR.exists():
    print("\\nAvailable Kaggle inputs:")
    for item in sorted(KAGGLE_INPUT_DIR.iterdir()):
        print("-", item)


In [ ]:
import shutil
import zipfile
import yaml

def has_yolo_split_structure(folder: Path):
    train_ok = (folder / "train" / "images").exists() and (folder / "train" / "labels").exists()
    valid_ok = (folder / "valid" / "images").exists() and (folder / "valid" / "labels").exists()
    val_ok = (folder / "val" / "images").exists() and (folder / "val" / "labels").exists()
    return train_ok and (valid_ok or val_ok)

def find_dataset_root(base: Path):
    if has_yolo_split_structure(base):
        return base

    for folder in sorted(base.rglob("*")):
        if folder.is_dir() and has_yolo_split_structure(folder):
            return folder

    raise FileNotFoundError(
        f"Could not find YOLO structure under {base}. Expected train/images, train/labels and valid/images or val/images."
    )

def normalize_names(names, nc=None):
    if isinstance(names, list):
        return names
    if isinstance(names, dict):
        return [value for _, value in sorted(names.items(), key=lambda item: int(item[0]))]
    if nc is not None:
        return [f"class_{i}" for i in range(int(nc))]
    raise ValueError("Could not infer class names. Add names or nc to data.yaml before uploading the dataset to Kaggle.")

def choose_input_path():
    if DATASET_INPUT_PATH:
        path = Path(DATASET_INPUT_PATH)
        if not path.exists():
            raise FileNotFoundError(f"DATASET_INPUT_PATH does not exist: {path}")
        return path

    if not KAGGLE_INPUT_DIR.exists():
        raise FileNotFoundError("/kaggle/input does not exist. Add your dataset as a Kaggle input first.")

    candidates = sorted(KAGGLE_INPUT_DIR.rglob("*.zip")) + [p for p in sorted(KAGGLE_INPUT_DIR.iterdir()) if p.is_dir()]
    if not candidates:
        raise FileNotFoundError("No zip files or dataset folders found in /kaggle/input.")
    return candidates[0]

input_path = choose_input_path()
print("Using input:", input_path)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if input_path.is_file() and input_path.suffix.lower() == ".zip":
    with zipfile.ZipFile(input_path, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    dataset_root = find_dataset_root(EXTRACT_DIR)
else:
    dataset_root = find_dataset_root(input_path)

val_split_name = "valid" if (dataset_root / "valid" / "images").exists() else "val"
test_exists = (dataset_root / "test" / "images").exists()

yaml_candidates = [dataset_root / "data.yaml", dataset_root / "data.yml"]
existing_yaml = next((path for path in yaml_candidates if path.exists()), None)

raw_yaml = {}
if existing_yaml:
    raw_yaml = yaml.safe_load(existing_yaml.read_text(encoding="utf-8")) or {}

class_names = normalize_names(raw_yaml.get("names"), raw_yaml.get("nc"))

clean_yaml = {
    "path": str(dataset_root),
    "train": "train/images",
    "val": f"{val_split_name}/images",
    "nc": len(class_names),
    "names": class_names,
}

if test_exists:
    clean_yaml["test"] = "test/images"

clean_yaml_path = WORKDIR / "data.kaggle.yaml"
clean_yaml_path.write_text(
    yaml.safe_dump(clean_yaml, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print("Dataset root:", dataset_root)
print("Validation split:", val_split_name)
print("Clean YAML:\\n")
print(clean_yaml_path.read_text(encoding="utf-8"))


In [ ]:
import pandas as pd
from ultralytics import YOLO
from IPython.display import Image, display

# Choose yolo11n.pt, yolo11s.pt, yolo11m.pt, yolo11l.pt, or yolo11x.pt.
# Start with n/s on Kaggle T4 if you are unsure about VRAM.
MODEL_NAME = "yolo11s.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = -1
PATIENCE = 10
RUN_NAME = "rada_tpii_yolo11"

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(clean_yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=0,
    project=str(WORKDIR / "runs"),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)

run_dir = Path(results.save_dir)
best_weights = run_dir / "weights" / "best.pt"
last_weights = run_dir / "weights" / "last.pt"

dest_dir = RESULTS_DIR / RUN_NAME
dest_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_weights, dest_dir / "best.pt")
shutil.copy2(last_weights, dest_dir / "last.pt")
shutil.copy2(clean_yaml_path, dest_dir / "data.kaggle.yaml")

print("Run directory:", run_dir)
print("Saved best.pt to:", dest_dir / "best.pt")
print("Saved last.pt to:", dest_dir / "last.pt")

if not test_exists:
    raise FileNotFoundError(
        "No se encontro test/images en el dataset. Anade un split test con imagenes y etiquetas para generar sus graficas."
    )

best_model = YOLO(str(best_weights))
test_metrics = best_model.val(
    data=str(clean_yaml_path),
    split="test",
    device=0,
    plots=True,
    project=str(WORKDIR / "runs"),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)
test_eval_dir = Path(test_metrics.save_dir)

print("Test metrics:")
print(test_metrics.results_dict)
print("Test evaluation directory:", test_eval_dir)

test_results_csv = dest_dir / "test_results.csv"
pd.DataFrame([test_metrics.results_dict]).to_csv(test_results_csv, index=False)
print("Saved test results CSV to:", test_results_csv)

training_results_csv = run_dir / "results.csv"
if training_results_csv.exists():
    shutil.copy2(training_results_csv, dest_dir / "training_results.csv")
    print("Saved training results CSV to:", dest_dir / "training_results.csv")

test_plot_names = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "PR_curve.png",
]

for plot_name in test_plot_names:
    plot_path = test_eval_dir / plot_name
    if plot_path.exists():
        print(plot_name)
        display(Image(filename=str(plot_path), width=800))

preview_source = dataset_root / "test" / "images"
best_model.predict(
    source=str(preview_source),
    conf=0.25,
    save=True,
    project=str(WORKDIR / "predictions"),
    name="test_preview",
    exist_ok=True,
)

preview_dir = WORKDIR / "predictions" / "test_preview"
for image_path in sorted(preview_dir.glob("*"))[:5]:
    display(Image(filename=str(image_path), width=600))

test_archive_path = shutil.make_archive(str(dest_dir / "test_evaluation"), "zip", root_dir=str(test_eval_dir))
print("Saved test evaluation archive to:", test_archive_path)

archive_path = shutil.make_archive(str(dest_dir / "run_artifacts"), "zip", root_dir=str(run_dir))
print("Saved run archive to:", archive_path)
